# Build an Epsilon-Greedy Bandit from Scratch in 5 Lines

**Goal:** Implement and test epsilon-greedy exploration on a commodity sector allocation problem.

**Time:** 15 minutes

**You'll learn:**
- How epsilon-greedy balances exploration and exploitation
- The impact of ε on regret and arm selection
- When to use decaying vs fixed ε

## Setup: Import Libraries

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# Plotting config
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
apply_course_theme()
apply_plot_theme()

## Define a Multi-Armed Bandit Environment

We'll simulate 5 commodity sectors with different expected returns:
- **Arm 0 (Energy):** μ = 0.3, σ = 0.5
- **Arm 1 (Metals):** μ = 0.5, σ = 0.3
- **Arm 2 (Agriculture):** μ = 0.8, σ = 0.2 ← **Best arm**
- **Arm 3 (Livestock):** μ = 0.4, σ = 0.4
- **Arm 4 (Softs):** μ = 0.2, σ = 0.6

Rewards are normalized to [0, 1] (representing normalized daily returns).

In [ ]:
class CommodityBandit:
    """Multi-armed bandit with Gaussian rewards"""
    def __init__(self, means, stds):
        self.means = np.array(means)
        self.stds = np.array(stds)
        self.k = len(means)
        self.best_arm = np.argmax(means)
        self.best_mean = np.max(means)
    
    def pull(self, arm):
        """Pull an arm and receive a noisy reward"""
        reward = np.random.normal(self.means[arm], self.stds[arm])
        return np.clip(reward, 0, 1)  # Keep in [0, 1]
    
    def get_regret(self, arm):
        """Instantaneous regret for pulling this arm"""
        return self.best_mean - self.means[arm]

# Create environment
sector_names = ['Energy', 'Metals', 'Agriculture', 'Livestock', 'Softs']
bandit = CommodityBandit(
    means=[0.3, 0.5, 0.8, 0.4, 0.2],
    stds=[0.5, 0.3, 0.2, 0.4, 0.6]
)

print(f"Best arm: {bandit.best_arm} ({sector_names[bandit.best_arm]})")
print(f"Expected reward: {bandit.best_mean:.2f}")

## Implement Epsilon-Greedy Agent (~10 lines)

The algorithm:
1. With probability ε: choose a random arm (explore)
2. With probability 1-ε: choose the arm with highest estimated value (exploit)
3. Update the value estimate with the observed reward

In [ ]:
class EpsilonGreedyAgent:
    def __init__(self, k_arms, epsilon=0.1):
        self.k = k_arms
        self.epsilon = epsilon
        self.q_estimates = np.zeros(k_arms)  # Estimated values Q̂(a)
        self.action_counts = np.zeros(k_arms)  # Number of pulls N(a)
    
    def select_action(self):
        """Select action using epsilon-greedy policy"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.k)  # Explore
        else:
            return np.argmax(self.q_estimates)  # Exploit
    
    def update(self, action, reward):
        """Update value estimate for the selected action"""
        self.action_counts[action] += 1
        n = self.action_counts[action]
        # Incremental mean update: Q_new = Q_old + (R - Q_old)/n
        self.q_estimates[action] += (reward - self.q_estimates[action]) / n

# Create agent
agent = EpsilonGreedyAgent(k_arms=5, epsilon=0.1)
print(f"Agent initialized with ε = {agent.epsilon}")

## Run 10,000 Steps and Track Performance

We'll track:
- Cumulative reward (how much we earned)
- Cumulative regret (how much we lost vs always picking the best arm)
- Action selection counts (which arms we explored)

In [ ]:
T = 10000
rewards = np.zeros(T)
regrets = np.zeros(T)
actions = np.zeros(T, dtype=int)

for t in range(T):
    # Agent selects action
    action = agent.select_action()
    
    # Environment provides reward
    reward = bandit.pull(action)
    
    # Agent updates estimates
    agent.update(action, reward)
    
    # Track performance
    rewards[t] = reward
    regrets[t] = bandit.get_regret(action)
    actions[t] = action

print(f"Total reward: {rewards.sum():.2f}")
print(f"Total regret: {regrets.sum():.2f}")
print(f"Average reward per step: {rewards.mean():.3f}")

## Plot Cumulative Reward

Cumulative reward shows how much value we've accumulated over time.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot cumulative reward
plt.subplot(1, 2, 1)
plt.plot(np.cumsum(rewards), label='Epsilon-Greedy', linewidth=2)
plt.plot(np.cumsum([bandit.best_mean] * T), 'k--', label='Optimal (always best arm)', alpha=0.7)
plt.xlabel('Time Step')
plt.ylabel('Cumulative Reward')
plt.title(f'Cumulative Reward (ε={agent.epsilon})')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot cumulative regret
plt.subplot(1, 2, 2)
plt.plot(np.cumsum(regrets), linewidth=2, color='red')
plt.xlabel('Time Step')
plt.ylabel('Cumulative Regret')
plt.title(f'Cumulative Regret (ε={agent.epsilon})')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final cumulative regret: {np.cumsum(regrets)[-1]:.2f}")

## Plot Per-Arm Pull Counts

This shows how the agent explored different commodity sectors.

In [ ]:
pull_counts = agent.action_counts
pull_percentages = 100 * pull_counts / T

plt.figure(figsize=(10, 6))
bars = plt.bar(sector_names, pull_counts, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'])

# Highlight the best arm
bars[bandit.best_arm].set_edgecolor('black')
bars[bandit.best_arm].set_linewidth(3)

plt.ylabel('Number of Pulls')
plt.title(f'Arm Selection Frequency (ε={agent.epsilon}, T={T})')
plt.grid(True, alpha=0.3, axis='y')

# Add percentage labels on bars
for i, (count, pct) in enumerate(zip(pull_counts, pull_percentages)):
    plt.text(i, count + 100, f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.show()

print("\nPull counts by sector:")
for i, name in enumerate(sector_names):
    star = " ⭐ BEST" if i == bandit.best_arm else ""
    print(f"  {name:12s}: {int(pull_counts[i]):5d} pulls ({pull_percentages[i]:5.2f}%){star}")

print(f"\nExpected exploration per arm: ~{agent.epsilon * T / bandit.k:.0f} pulls")
print(f"Expected exploitation of best: ~{(1 - agent.epsilon) * T:.0f} pulls")

## Final Value Estimates vs True Means

Let's see how well the agent learned the true reward distribution.

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(len(sector_names))
width = 0.35

plt.bar(x - width/2, bandit.means, width, label='True Mean', alpha=0.7, color='steelblue')
plt.bar(x + width/2, agent.q_estimates, width, label='Estimated Q̂', alpha=0.7, color='coral')

plt.xlabel('Commodity Sector')
plt.ylabel('Reward')
plt.title('True Means vs Learned Estimates')
plt.xticks(x, sector_names)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print("\nLearning accuracy:")
for i, name in enumerate(sector_names):
    error = abs(agent.q_estimates[i] - bandit.means[i])
    print(f"  {name:12s}: True={bandit.means[i]:.3f}, Estimated={agent.q_estimates[i]:.3f}, Error={error:.3f}")

## 🔧 Modify This: Change ε and Observe the Impact

Try different values of ε:
- **ε = 0.0:** Pure exploitation (no exploration)
- **ε = 0.01:** Very low exploration (1%)
- **ε = 0.1:** Balanced (10% exploration)
- **ε = 0.5:** High exploration (50% random)
- **ε = 1.0:** Pure exploration (random search)

What happens to:
- Cumulative regret?
- Pull count distribution?
- Estimate accuracy?

In [ ]:
# Experiment with different epsilon values
epsilon_values = [0.0, 0.01, 0.1, 0.3, 0.5, 1.0]
results = []

for eps in epsilon_values:
    agent = EpsilonGreedyAgent(k_arms=5, epsilon=eps)
    total_regret = 0
    
    for t in range(T):
        action = agent.select_action()
        reward = bandit.pull(action)
        agent.update(action, reward)
        total_regret += bandit.get_regret(action)
    
    results.append({
        'epsilon': eps,
        'regret': total_regret,
        'best_arm_pct': 100 * agent.action_counts[bandit.best_arm] / T
    })

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot([r['epsilon'] for r in results], [r['regret'] for r in results], 'o-', linewidth=2, markersize=8)
plt.xlabel('Epsilon (ε)')
plt.ylabel('Total Regret')
plt.title('Impact of ε on Regret')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([r['epsilon'] for r in results], [r['best_arm_pct'] for r in results], 'o-', linewidth=2, markersize=8, color='green')
plt.xlabel('Epsilon (ε)')
plt.ylabel('% Pulls on Best Arm')
plt.title('Impact of ε on Exploitation')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEpsilon sensitivity analysis:")
print(f"{'ε':>6s}  {'Regret':>8s}  {'Best Arm %':>10s}")
print("-" * 30)
for r in results:
    print(f"{r['epsilon']:>6.2f}  {r['regret']:>8.1f}  {r['best_arm_pct']:>10.1f}%")

## 🔧 Modify This: Add Decaying Epsilon

Fixed ε wastes pulls on random exploration forever. Let's try decaying ε over time:

$$\epsilon(t) = \min\left(1, \frac{C}{\sqrt{t+1}}\right)$$

This explores a lot early (when uncertain) and very little late (when confident).

In [ ]:
class DecayingEpsilonGreedy(EpsilonGreedyAgent):
    def __init__(self, k_arms, epsilon_fn=lambda t: 10 / np.sqrt(t + 1)):
        super().__init__(k_arms, epsilon=1.0)
        self.epsilon_fn = epsilon_fn
        self.t = 0
    
    def select_action(self):
        self.epsilon = min(1.0, self.epsilon_fn(self.t))
        self.t += 1
        return super().select_action()

# Compare fixed vs decaying
agent_fixed = EpsilonGreedyAgent(k_arms=5, epsilon=0.1)
agent_decay = DecayingEpsilonGreedy(k_arms=5)

regrets_fixed = []
regrets_decay = []
epsilon_schedule = []

for t in range(T):
    # Fixed epsilon
    a1 = agent_fixed.select_action()
    r1 = bandit.pull(a1)
    agent_fixed.update(a1, r1)
    regrets_fixed.append(bandit.get_regret(a1))
    
    # Decaying epsilon
    a2 = agent_decay.select_action()
    r2 = bandit.pull(a2)
    agent_decay.update(a2, r2)
    regrets_decay.append(bandit.get_regret(a2))
    epsilon_schedule.append(agent_decay.epsilon)

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(np.cumsum(regrets_fixed), label='Fixed ε=0.1', linewidth=2)
plt.plot(np.cumsum(regrets_decay), label='Decaying ε(t)=10/√t', linewidth=2)
plt.xlabel('Time Step')
plt.ylabel('Cumulative Regret')
plt.title('Fixed vs Decaying Epsilon')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epsilon_schedule, linewidth=2, color='purple')
plt.xlabel('Time Step')
plt.ylabel('Epsilon Value')
plt.title('Epsilon Decay Schedule')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal regret comparison:")
print(f"  Fixed ε=0.1:        {np.sum(regrets_fixed):.2f}")
print(f"  Decaying ε(t):      {np.sum(regrets_decay):.2f}")
print(f"  Improvement:        {(1 - np.sum(regrets_decay)/np.sum(regrets_fixed))*100:.1f}%")

## 🔧 Modify This: Try Different Arm Distributions

Change the bandit environment:
- Make gaps smaller (all arms close to 0.5)
- Make gaps larger (one arm much better)
- Add more arms (K=10 or K=20)
- Make rewards non-stationary (change means over time)

In [ ]:
# Example: Small gaps (harder problem)
bandit_hard = CommodityBandit(
    means=[0.48, 0.49, 0.50, 0.49, 0.48],  # All very similar
    stds=[0.2, 0.2, 0.2, 0.2, 0.2]
)

agent = EpsilonGreedyAgent(k_arms=5, epsilon=0.1)
regrets_hard = []

for t in range(T):
    action = agent.select_action()
    reward = bandit_hard.pull(action)
    agent.update(action, reward)
    regrets_hard.append(bandit_hard.get_regret(action))

print("\nHard problem (small gaps):")
print(f"  True means: {bandit_hard.means}")
print(f"  Max gap: {bandit_hard.means.max() - bandit_hard.means.min():.3f}")
print(f"  Total regret: {np.sum(regrets_hard):.2f}")
print(f"\nPull distribution:")
for i in range(5):
    print(f"    Arm {i}: {int(agent.action_counts[i])} pulls ({100*agent.action_counts[i]/T:.1f}%)")

## Summary

### What We Built
- ✅ Epsilon-greedy bandit from scratch in ~10 lines
- ✅ Tested on commodity sector allocation problem
- ✅ Visualized exploration patterns and regret
- ✅ Compared fixed vs decaying ε

### Key Takeaways

1. **Epsilon controls exploration-exploitation tradeoff**
   - Too low → premature convergence (miss best arm)
   - Too high → wasteful random search
   - Rule of thumb: ε ∈ [0.05, 0.2]

2. **Decaying ε beats fixed ε**
   - Explore a lot early (when uncertain)
   - Exploit more late (when confident)
   - Typical schedule: ε(t) = C/√t

3. **Problem difficulty matters**
   - Small gaps → need more exploration (higher ε or longer T)
   - Large gaps → can exploit earlier (lower ε)

### What's Next?

Epsilon-greedy is simple but has limitations:
- Random exploration is wasteful (treats bad arms equally)
- Fixed ε leads to linear regret (εT term)
- Needs hyperparameter tuning

**Next notebook:** We'll implement UCB1, which uses confidence bounds instead of random exploration to achieve logarithmic regret.

**Go deeper:** Read [guides/01_epsilon_greedy.md](../guides/01_epsilon_greedy.md) for theory and regret bounds.

In [ ]:
key_takeaways(["Epsilon controls exploration-exploitation tradeoff", "Decaying ε beats fixed ε", "Problem difficulty matters"])